# NID OCR benchmark — Tesseract vs EasyOCR vs PaddleOCR

A timer is used. The timer starts before the image bytes are decoded and stops after the fields are
parsed, which is everything a server does for one request. Network and HTTP overhead
are not included.

Each engine runs in its own virtual environment, as a separate process, driven by a
worker script in `bench/`. They all share `bench/common.py`, so image decoding, the
field parser and the scoring are identical — the only difference between engines is
the OCR call itself.

**Run the cells in order.** Each section writes its raw results to
`results/<run id>/` before anything is scored, so the analysis can be redone without
re-running the OCR.

| Section | Measures |
|---|---|
| 1 | Setup and environment check |
| 2 | Accuracy (the filter — a fast, cheap engine that misreads NID numbers is out) |
| 3 | Speed: latency p50 / p95 and cold start |
| 4 | Throughput: the TPS scaling test |
| 5 | Hardware: memory and CPU per worker |
| 6 | Cost: servers and money at 100–1000 TPS |
| 7 | Summary table and conclusion |

## 1. Setup

### 1.1 Configuration

Everything you would want to change lives in this one cell.

In [4]:
import os, sys, json, math, time, platform, subprocess, threading
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt

# --- Paths -----------------------------------------------------------------
# The kernel's working directory is not always the project folder, so find the
# project by looking for bench/common.py in cwd and its parents.
def find_root(start=None):
    cands = []
    if start:
        cands.append(Path(start))
    try:
        cands.append(Path(globals()["__vsc_ipynb_file__"]).parent)   # VS Code
    except Exception:
        pass
    cands.append(Path.cwd())
    seen = []
    for c in cands:
        c = Path(c).resolve()
        for d in (c, *c.parents):
            seen.append(d)
            if (d / "bench" / "common.py").exists() and (d / "data").exists():
                return d
    raise FileNotFoundError(
        "Could not find the OCR-RnD folder (needs bench/common.py and data/).\n"
        "Set it by hand:  ROOT = Path(r'C:\\Users\\sadman.khan\\Documents\\OCR-RnD')\n"
        "Looked in:\n  " + "\n  ".join(str(s) for s in dict.fromkeys(seen)))

ROOT   = find_root()
BENCH  = ROOT / "bench"
DATA   = ROOT / "data"
IMAGES = DATA / "images"
TRUTH  = DATA / "data.csv"

sys.path.insert(0, str(BENCH))
import common                        # the shared parser / scorer

RUN_ID  = dt.datetime.now().strftime("%Y%m%d_%H%M")
OUTDIR  = ROOT / "results" / RUN_ID
OUTDIR.mkdir(parents=True, exist_ok=True)


def venv_python(folder):
    """Find the interpreter inside a venv on Windows or Linux/mac."""
    base = ROOT / folder
    for cand in (base / "Scripts" / "python.exe", base / "bin" / "python"):
        if cand.exists():
            return cand
    raise FileNotFoundError(f"no python found in {base}")


# --- What we are benchmarking ----------------------------------------------
# PaddleOCR appears twice: it defaults to its largest model tier, so the small
# tier is measured too. That answers "is the big model worth its cost on digits?"
ENGINES = [
    {"name": "tesseract",       "venv": ".venv",         "script": "worker_tesseract.py", "env": {}},
    {"name": "easyocr",         "venv": ".venv-easyocr", "script": "worker_easyocr.py",   "env": {}},
    {"name": "paddle-medium",   "venv": ".venv-paddle",  "script": "worker_paddle.py",    "env": {}},
    {"name": "paddle-small",    "venv": ".venv-paddle",  "script": "worker_paddle.py",
     "env": {"BENCH_PADDLE_DET": "PP-OCRv6_small_det",
             "BENCH_PADDLE_REC": "PP-OCRv6_small_rec"}},
]
for e in ENGINES:
    e["python"] = venv_python(e["venv"])

# --- Benchmark settings -----------------------------------------------------
THREADS_PER_WORKER = 1      # 1 worker = 1 core, so the capacity maths is simple
LATENCY_REPEATS    = 5      # passes over the image set (20 images x 5 = 100 samples)
TPS_DURATION       = 60     # seconds per throughput run
WARMUP_REQUESTS    = 3      # untimed requests before measuring

LOGICAL_CORES  = psutil.cpu_count(logical=True)
PHYSICAL_CORES = psutil.cpu_count(logical=False) or LOGICAL_CORES

# Worker counts for the scaling test: 1, 2, 4, ... up to the logical core count.
WORKER_COUNTS = sorted({1, 2, 4, LOGICAL_CORES})
WORKER_COUNTS = [w for w in WORKER_COUNTS if w <= LOGICAL_CORES]

# --- Capacity and cost ------------------------------------------------------
TPS_TARGETS = [100, 300, 1000]
UTILISATION = 0.70          # plan servers to run 70% busy; at 100% queues explode

# VERIFY THESE ON THE AWS PRICING CALCULATOR BEFORE THE REPORT GOES OUT.
# Placeholder is the us-east-1 on-demand rate; Mumbai/Singapore run higher.
SERVERS = [
    {"name": "c7i.2xlarge", "vcpu": 8,  "ram_gb": 16, "usd_per_hour": 0.357},
    {"name": "c7i.4xlarge", "vcpu": 16, "ram_gb": 32, "usd_per_hour": 0.714},
]
HOURS_PER_MONTH = 730

pd.set_option("display.width", 200, "display.max_columns", 50)
print(f"project root  : {ROOT}")
print(f"run id        : {RUN_ID}")
print(f"results       : {OUTDIR}")
print(f"machine       : {platform.system()} {platform.release()}, "
      f"{PHYSICAL_CORES} physical / {LOGICAL_CORES} logical cores, "
      f"{psutil.virtual_memory().total/1e9:.1f} GB RAM")
print(f"images        : {len(list(IMAGES.glob('*')))} files in {IMAGES}")
print(f"scaling test  : {WORKER_COUNTS} workers x {TPS_DURATION}s")

project root  : C:\Users\sadman.khan\Documents\OCR-RnD
run id        : 20260921_1622
results       : C:\Users\sadman.khan\Documents\OCR-RnD\results\20260921_1622
machine       : Windows 11, 4 physical / 8 logical cores, 16.9 GB RAM
images        : 5 files in C:\Users\sadman.khan\Documents\OCR-RnD\data\images
scaling test  : [1, 2, 4, 8] workers x 60s


### 1.2 Helpers

`launch()` starts one or more worker processes and watches their memory and CPU
while they run. Three details matter for the numbers to be trustworthy:

- **Thread pinning.** Each worker is held to one thread, so "workers" and "cores"
  mean the same thing in the scaling test. Without this the engines fight each
  other for cores and the curve is meaningless.
- **A shared start time.** Model loading takes seconds (EasyOCR ~11 s). Workers
  load, then wait for a common clock time before starting. Otherwise the fast
  loaders do all their work before the slow ones have begun.
- **Child processes are counted.** If Tesseract falls back to `pytesseract` it
  spawns `tesseract.exe` per request; that memory belongs to the worker.

In [5]:
THREAD_ENV = {k: str(THREADS_PER_WORKER) for k in
              ("OMP_NUM_THREADS", "OMP_THREAD_LIMIT", "MKL_NUM_THREADS",
               "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS")}


class ResourceMonitor(threading.Thread):
    """Samples RSS and CPU% of the worker processes (and their children)."""

    def __init__(self, pids, interval=0.5):
        super().__init__(daemon=True)
        self.pids, self.interval = list(pids), interval
        self.samples, self._stopped = [], threading.Event()

    def run(self):
        procs = {}
        for pid in self.pids:
            try:
                p = psutil.Process(pid)
                p.cpu_percent(None)          # priming call; first reading is always 0
                procs[pid] = p
            except psutil.Error:
                pass
        while not self._stopped.is_set():
            now = time.time()
            for pid, p in list(procs.items()):
                try:
                    rss = p.memory_info().rss
                    cpu = p.cpu_percent(None)
                    for ch in p.children(recursive=True):
                        try:
                            rss += ch.memory_info().rss
                            cpu += ch.cpu_percent(None)
                        except psutil.Error:
                            pass
                    self.samples.append({"t": now, "pid": pid,
                                         "rss_mb": rss / 1e6, "cpu_pct": cpu})
                except psutil.Error:
                    procs.pop(pid, None)
            self._stopped.wait(self.interval)

    def stop(self):
        self._stopped.set()


def _meta_from(stdout, stderr, rc, cmd):
    """The worker prints one JSON line last. PaddleOCR prints noise before it."""
    for line in reversed((stdout or "").splitlines()):
        line = line.strip()
        if line.startswith("{"):
            try:
                return json.loads(line)
            except json.JSONDecodeError:
                continue
    raise RuntimeError(
        f"worker failed (exit {rc})\ncmd: {' '.join(map(str, cmd))}\n"
        f"--- stdout tail ---\n{(stdout or '')[-1500:]}\n"
        f"--- stderr tail ---\n{(stderr or '')[-1500:]}")


def launch(engine, mode, tag, n_workers=1, duration=None, repeats=None,
           start_lead=None, monitor=True):
    """
    Run `n_workers` copies of one engine's worker. Returns (metas, csv_paths, samples).
    Blocks until every worker exits.
    """
    env = dict(os.environ)
    env.update(THREAD_ENV)
    env.update({k: str(v) for k, v in engine["env"].items()})

    start_at = time.time() + (start_lead if start_lead is not None else 25.0)
    procs, outs, cmds = [], [], []

    for i in range(n_workers):
        out = OUTDIR / f"{mode}_{engine['name']}_{tag}_w{i}.csv"
        cmd = [str(engine["python"]), str(BENCH / engine["script"]),
               "--mode", mode, "--out", str(out), "--worker-id", str(i),
               "--warmup", str(WARMUP_REQUESTS),
               "--images", str(IMAGES), "--truth", str(TRUTH)]
        if duration is not None:
            cmd += ["--duration", str(duration), "--start-at", f"{start_at:.3f}"]
        if repeats is not None:
            cmd += ["--repeats", str(repeats)]
        procs.append(subprocess.Popen(cmd, env=env, cwd=str(ROOT),
                                      stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                      text=True, encoding="utf-8", errors="replace"))
        outs.append(out)
        cmds.append(cmd)

    mon = None
    if monitor:
        mon = ResourceMonitor([p.pid for p in procs])
        mon.start()

    metas = []
    for p, cmd in zip(procs, cmds):
        so, se = p.communicate()
        metas.append(_meta_from(so, se, p.returncode, cmd))

    samples = []
    if mon:
        mon.stop()
        mon.join(timeout=3)
        samples = mon.samples

    return metas, outs, samples


def read_csvs(paths):
    frames = [pd.read_csv(p, encoding="utf-8-sig", dtype=str) for p in paths if Path(p).exists()]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# Validated categorical palette; assigned in fixed order, never cycled.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
COLOR = {e["name"]: PALETTE[i % len(PALETTE)] for i, e in enumerate(ENGINES)}
GRID = dict(color="#d8d7d2", linewidth=0.8)

print("helpers ready")

helpers ready


### 1.3 Environment check

Loads each engine once and reports its version. Run this before anything long —
it catches a broken setup in seconds instead of twenty minutes in.

Check the Tesseract row says **`via tesserocr`**. If it says `via pytesseract`,
Tesseract is being charged process-startup time on every request that a real
server would never pay, which roughly doubles its measured latency.

In [6]:
check_rows, LOAD_SECONDS = [], {}

for e in ENGINES:
    metas, _, _ = launch(e, "check", "env", monitor=False)
    m = metas[0]
    LOAD_SECONDS[e["name"]] = m["load_seconds"]
    check_rows.append({
        "engine": e["name"],
        "version": m["version"],
        "cold start (s)": m["load_seconds"],
        "unsupported fields": ", ".join(m["unsupported_fields"]) or "—",
    })

env_df = pd.DataFrame(check_rows).set_index("engine")

# Workers must all be loaded before the synchronised start fires.
START_LEAD = round(max(LOAD_SECONDS.values()) * 1.6 + 8.0, 1)

(OUTDIR / "environment.json").write_text(json.dumps({
    "run_id": RUN_ID,
    "machine": {"system": platform.system(), "release": platform.release(),
                "physical_cores": PHYSICAL_CORES, "logical_cores": LOGICAL_CORES,
                "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
                "cpu": platform.processor()},
    "settings": {"threads_per_worker": THREADS_PER_WORKER,
                 "latency_repeats": LATENCY_REPEATS,
                 "tps_duration": TPS_DURATION,
                 "worker_counts": WORKER_COUNTS,
                 "utilisation": UTILISATION,
                 "start_lead": START_LEAD},
    "engines": check_rows,
}, indent=2), encoding="utf-8")

print(f"synchronised-start lead: {START_LEAD}s (slowest load "
      f"{max(LOAD_SECONDS.values()):.1f}s)\n")
env_df

synchronised-start lead: 81.0s (slowest load 45.6s)



,version,cold start (s),unsupported fields
engine,,,
tesseract,"tesseract 5.3.4 via tesserocr (lang=ben+eng, p...",0.221,—
easyocr,"easyocr 1.7.2 (langs=bn+en, gpu=False)",11.424,—
paddle-medium,"paddleocr 3.7.0 (3x API, lang=en, gpu=False, d...",6.542,"name_bn, father_bn, mother_bn"
paddle-small,"paddleocr 3.7.0 (3x API, lang=en, gpu=False, d...",45.614,"name_bn, father_bn, mother_bn"


## 2. Accuracy

Accuracy is a **filter, not a score**. An engine that is cheap and fast but misreads
NID numbers is disqualified however good its other numbers look, because a wrong NID
in a KYC pipeline is a compliance problem, not a quality problem.

Two measures per field:

- **Exact match** after normalising — this is what the pipeline actually needs, since
  a field is either right or it is wrong.
- **Character error rate (CER)** — how close the misses were. A CER of 0.1 on a
  10-digit NID means one wrong digit, which is a very different situation from an
  unreadable field, even though both score zero on exact match.

Fields an engine has no model for are shown as `n/a`, never as 0%.

In [ ]:
acc_files, acc_meta = {}, {}

for e in ENGINES:
    print(f"running {e['name']} ...", end=" ", flush=True)
    metas, outs, samples = launch(e, "accuracy", "run", start_lead=0)
    acc_files[e["name"]] = outs
    acc_meta[e["name"]] = metas[0]
    if samples:
        peak = max(s["rss_mb"] for s in samples)
        acc_meta[e["name"]]["peak_rss_mb"] = round(peak, 1)
    print(f"done ({metas[0]['images']} cards)")

truth = common.read_truth(TRUTH)

rows = []
for name, paths in acc_files.items():
    unsupported = set(acc_meta[name]["unsupported_fields"])
    df = read_csvs(paths).fillna("")
    for _, r in df.iterrows():
        t = truth.get(r["image_id"])
        if t is None:
            continue
        for f in common.FIELD_KEYS:
            ok = None if f in unsupported else common.exact_match(f, r[f], t[f])
            cer = np.nan if f in unsupported else common.char_error_rate(f, r[f], t[f])
            rows.append({"engine": name, "image_id": r["image_id"], "field": f,
                         "prediction": r[f], "truth": t[f],
                         "match": ok, "cer": cer, "status": r["status"]})

scored = pd.DataFrame(rows)
scored.to_csv(OUTDIR / "accuracy_scored.csv", index=False, encoding="utf-8-sig")
print(f"\n{len(scored)} field comparisons -> accuracy_scored.csv")

running tesseract ... done (20 cards)
running easyocr ... 

In [ ]:
def _fmt_pct(v):
    return "n/a" if pd.isna(v) else f"{v*100:.0f}%"

exact = (scored.pivot_table(index="engine", columns="field", values="match",
                            aggfunc="mean", dropna=False)
                .reindex(columns=common.FIELD_KEYS)
                .reindex([e["name"] for e in ENGINES]))

cer = (scored.pivot_table(index="engine", columns="field", values="cer",
                          aggfunc="mean", dropna=False)
             .reindex(columns=common.FIELD_KEYS)
             .reindex([e["name"] for e in ENGINES]))

print("Field-level exact match")
display(exact.map(_fmt_pct))

print("\nMean character error rate (0.00 = perfect, n/a = no model for this field)")
display(cer.map(lambda v: "n/a" if pd.isna(v) else f"{v:.3f}"))

# The two fields that decide compliance.
critical = (scored[scored["field"].isin(["nid", "dob"])]
            .pivot_table(index="engine", columns="field", values="match",
                         aggfunc="mean", dropna=False)
            .reindex([e["name"] for e in ENGINES]))
print("\nCompliance-critical fields")
display(critical.map(_fmt_pct))

### 2.1 Look at the failures

The numbers tell you *that* something failed; only the raw text tells you *why*.

The parser finds names by their label (`নাম`, `Name`, `পিতা`, `মাতা`). If an engine
read a name correctly but missed its label, the field still scores zero — that is a
parser limitation, not an engine limitation, and the fix is a better parser, not a
different engine. Date and NID are found by pattern instead of label, so they do not
have this weakness.

Run this before drawing any conclusion about a low score.

In [ ]:
misses = scored[(scored["match"] == False)].copy()

print(f"{len(misses)} field misses\n")
print("Misses by engine and field:")
display(misses.pivot_table(index="engine", columns="field",
                           values="image_id", aggfunc="count").fillna(0).astype(int))

# Split "read it wrong" from "returned nothing" - they have different causes.
misses["kind"] = np.where(misses["prediction"].str.strip() == "",
                          "empty (not found)", "wrong text")
print("\nMiss type:")
display(misses.pivot_table(index="engine", columns="kind",
                           values="image_id", aggfunc="count").fillna(0).astype(int))

print("\nWorst 15 misses by character error rate:")
display(misses.nlargest(15, "cer")[["engine", "image_id", "field",
                                    "truth", "prediction", "cer"]])

In [ ]:
# Raw OCR text for one card, per engine. This is the diagnostic that tells you
# whether a missing field was never read, or was read but not labelled.
INSPECT_ID = scored["image_id"].iloc[0]      # change to any image_id

for name, paths in acc_files.items():
    df = read_csvs(paths).fillna("")
    row = df[df["image_id"] == INSPECT_ID]
    if row.empty:
        continue
    print("=" * 70)
    print(f"{name}  |  card {INSPECT_ID}  |  {row.iloc[0]['ms']} ms")
    print("=" * 70)
    print(row.iloc[0]["raw_text"][:1200])
    print()

## 3. Speed

Single worker, one thread, warmed up. Warm-up matters: the first few requests on any
of these engines include lazy setup and can be many times slower than steady state,
so measuring them would just measure start-up.

**p50** is a typical request. **p95** means 95% of requests finish faster than this —
it is the number that tells you how bad the slow cases get, and it is what users
actually feel. The average is not reported, because one slow outlier drags it
somewhere that describes no real request.

Cold start is reported separately: it happens once when a server boots, and it
matters for autoscaling, not for a single request.

In [ ]:
lat_frames, lat_res = {}, {}

for e in ENGINES:
    print(f"running {e['name']} ...", end=" ", flush=True)
    metas, outs, samples = launch(e, "latency", "run",
                                  repeats=LATENCY_REPEATS, start_lead=0)
    df = read_csvs(outs)
    df["ms"] = pd.to_numeric(df["ms"], errors="coerce")
    lat_frames[e["name"]] = df
    lat_res[e["name"]] = samples
    print(f"done ({len(df)} requests)")

lat_all = pd.concat([d.assign(engine=n) for n, d in lat_frames.items()], ignore_index=True)
lat_all.to_csv(OUTDIR / "latency_raw.csv", index=False, encoding="utf-8-sig")

lat_rows = []
for e in ENGINES:
    n = e["name"]
    ok = lat_frames[n][lat_frames[n]["status"] == "ok"]["ms"].dropna()
    lat_rows.append({
        "engine": n,
        "n": len(ok),
        "p50 ms": round(ok.quantile(0.50), 1),
        "p95 ms": round(ok.quantile(0.95), 1),
        "min ms": round(ok.min(), 1),
        "max ms": round(ok.max(), 1),
        "cards/sec (1 core)": round(1000 / ok.quantile(0.50), 2),
        "cold start s": LOAD_SECONDS[n],
    })

lat_df = pd.DataFrame(lat_rows).set_index("engine")
lat_df.to_csv(OUTDIR / "latency_summary.csv", encoding="utf-8-sig")
lat_df

In [ ]:
names = [e["name"] for e in ENGINES]
fig, ax = plt.subplots(figsize=(9, 1.5 + 0.75 * len(names)))
y = np.arange(len(names))

ax.barh(y - 0.19, [lat_df.loc[n, "p50 ms"] for n in names], height=0.34,
        color=[COLOR[n] for n in names], label="p50")
ax.barh(y + 0.19, [lat_df.loc[n, "p95 ms"] for n in names], height=0.34,
        color=[COLOR[n] for n in names], alpha=0.45, label="p95")

for i, n in enumerate(names):                       # direct labels (relief rule)
    for off, col in ((-0.19, "p50 ms"), (0.19, "p95 ms")):
        v = lat_df.loc[n, col]
        ax.text(v * 1.02, i + off, f"{v:.0f}", va="center", fontsize=9, color="#52514e")

ax.set_yticks(y, names)
ax.invert_yaxis()
ax.set_xlabel("milliseconds per card (lower is better)")
ax.set_title("Request latency, single worker on one core", loc="left", fontsize=11)
ax.xaxis.grid(True, **GRID)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_xlim(0, max(lat_df["p95 ms"]) * 1.12)
plt.tight_layout()
plt.savefig(OUTDIR / "latency.png", dpi=150)
plt.show()

## 4. Throughput (TPS)

You cannot generate 100–1000 TPS on a laptop, so this is done in two parts.

**Part A, measured.** Run the same engine with 1, then 2, then 4 workers at once,
all starting together, each looping over cards for a fixed time. Total cards divided
by elapsed time gives the machine's throughput at that worker count. The curve rises
and then flattens — where it flattens, the machine is saturated.

**Part B, calculated.** Turn "cards per second per core" into the servers needed for
a target rate. That happens in section 6.

Each worker is pinned to one thread, so one worker is one core and the numbers scale
predictably onto a cloud instance.

⚠️ This is the long cell. Roughly `len(ENGINES) × len(WORKER_COUNTS) × (TPS_DURATION + start lead)`
seconds — typically 20–40 minutes. Close other applications first; background load
will distort the results.

In [ ]:
tps_rows, tps_res = [], {}

for e in ENGINES:
    for n_workers in WORKER_COUNTS:
        print(f"{e['name']:<16} {n_workers} worker(s) ...", end=" ", flush=True)
        metas, outs, samples = launch(e, "throughput", f"n{n_workers}",
                                      n_workers=n_workers, duration=TPS_DURATION,
                                      start_lead=START_LEAD)

        completed = sum(m["completed"] for m in metas)
        elapsed = [m["elapsed_seconds"] for m in metas]
        mean_elapsed = float(np.mean(elapsed))
        cards_per_sec = completed / mean_elapsed

        df = read_csvs(outs)
        df["ms"] = pd.to_numeric(df["ms"], errors="coerce")
        ok = df[df["status"] == "ok"]["ms"].dropna()

        # Did the workers really run at the same time? If the spread is wide the
        # synchronised start failed and this row is not a valid parallel measurement.
        spread = (max(elapsed) - min(elapsed)) / mean_elapsed
        if spread > 0.15:
            print(f"WARNING: worker runtimes differ by {spread*100:.0f}% — "
                  f"increase START_LEAD", end=" ")

        peak_rss = max((s["rss_mb"] for s in samples), default=np.nan)
        tot_rss = np.nan
        if samples:
            by_t = pd.DataFrame(samples).groupby("t")["rss_mb"].sum()
            tot_rss = by_t.max()

        tps_rows.append({
            "engine": e["name"], "workers": n_workers,
            "cards": completed,
            "cards/sec": round(cards_per_sec, 2),
            "p50 ms": round(ok.quantile(0.50), 1) if len(ok) else np.nan,
            "p95 ms": round(ok.quantile(0.95), 1) if len(ok) else np.nan,
            "peak rss/worker MB": round(peak_rss, 1),
            "peak rss total MB": round(tot_rss, 1) if not np.isnan(tot_rss) else np.nan,
        })
        tps_res[(e["name"], n_workers)] = samples
        print(f"{cards_per_sec:.2f} cards/sec")

tps_df = pd.DataFrame(tps_rows)
tps_df.to_csv(OUTDIR / "throughput.csv", index=False, encoding="utf-8-sig")
tps_df

In [ ]:
pivot = tps_df.pivot(index="workers", columns="engine", values="cards/sec")
pivot = pivot.reindex(columns=[e["name"] for e in ENGINES])

fig, ax = plt.subplots(figsize=(8, 4.4))
ends = []
for name in pivot.columns:
    s = pivot[name].dropna()
    if s.empty:
        continue
    ax.plot(s.index, s.values, marker="o", markersize=7, linewidth=2,
            color=COLOR[name], label=name)
    ends.append((s.index[-1], s.values[-1], name))

ideal = pivot.iloc[0].max() * np.array(WORKER_COUNTS)
ax.plot(WORKER_COUNTS, ideal, linestyle=(0, (4, 4)), linewidth=1.4,
        color="#8a8983", label="perfect scaling", zorder=0)

ax.set_xlabel("concurrent workers (1 worker = 1 core)")
ax.set_ylabel("cards per second")
ax.set_title("Throughput scaling — where the line flattens, the machine is full",
             loc="left", fontsize=11)
ax.set_xticks(WORKER_COUNTS)
ax.grid(True, **GRID)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend(frameon=False, loc="upper left")
# Direct labels sit past the last point, so reserve horizontal room for them.
_span = max(WORKER_COUNTS) - min(WORKER_COUNTS) or 1
ax.set_xlim(min(WORKER_COUNTS) - _span * 0.06, max(WORKER_COUNTS) + _span * 0.38)

# Direct labels, nudged apart so engines with similar throughput stay readable.
_y0, _y1 = ax.get_ylim()
_gap, _placed = (_y1 - _y0) * 0.055, []
for _x, _y, _name in sorted(ends, key=lambda r: -r[1]):
    _ty = _y
    for _prev in _placed:
        if abs(_ty - _prev) < _gap:
            _ty = _prev - _gap
    _placed.append(_ty)
    ax.annotate(f"  {_name}", xy=(_x, _ty), fontsize=9, color=COLOR[_name], va="center")
plt.tight_layout()
plt.savefig(OUTDIR / "throughput_scaling.png", dpi=150)
plt.show()

print("Cards per second by worker count:")
display(pivot.round(2))

## 5. Hardware requirement

Two things decide how many workers fit on a server:

- **Memory per worker (RSS).** Neural engines hold their weights in memory; Tesseract
  is far lighter. This usually sets the limit before CPU does.
- **CPU per worker.** Confirms the thread pinning worked. A worker should sit near
  100% of one core. Much above that means it is using more than one core and the
  scaling numbers understate the hardware it really needs.

Model size on disk is also recorded, because it drives container image size and how
fast a new instance can come up during a traffic spike.

In [ ]:
hw_rows = []
for e in ENGINES:
    name = e["name"]
    single = tps_df[(tps_df["engine"] == name) & (tps_df["workers"] == 1)]
    samples = tps_res.get((name, 1), [])
    cpu = np.nan
    if samples:
        s = pd.DataFrame(samples)
        cpu = s[s["cpu_pct"] > 0]["cpu_pct"].median()
    hw_rows.append({
        "engine": name,
        "peak RSS / worker MB": float(single["peak rss/worker MB"].iloc[0]) if len(single) else np.nan,
        "median CPU % (1 worker)": round(cpu, 0) if not np.isnan(cpu) else np.nan,
        "cold start s": LOAD_SECONDS[name],
    })

hw_df = pd.DataFrame(hw_rows).set_index("engine")
hw_df["workers per 16 GB"] = (16 * 1024 / hw_df["peak RSS / worker MB"]).round(0)
hw_df.to_csv(OUTDIR / "hardware.csv", encoding="utf-8-sig")

print("If median CPU% is far above 100, thread pinning did not take effect.\n")
hw_df

## 6. Cost

Cost is calculated, not measured. It combines the throughput from section 4 with the
memory from section 5 and a server price.

```
vCPUs needed   = target TPS / (cards per sec per vCPU x utilisation)
servers by CPU = vCPUs needed / vCPUs per server
servers by RAM = vCPUs needed / workers that fit in the server's memory
servers needed = whichever is larger
```

Two honest caveats to carry into the report:

1. **A cloud vCPU is one hyperthread, and usually slower than a laptop core.** These
   figures are estimates. The way to fix that is to rerun section 4 on one real cloud
   instance; the worker design makes that a copy-and-run job.
2. **The 70% utilisation assumption is doing real work.** Sizing for 100% looks 30%
   cheaper and falls over during any spike, because a queue forms and p95 latency
   climbs sharply.

Prices below are placeholders. Check them on the AWS Pricing Calculator for your
region on the day you write the report.

In [ ]:
per_vcpu = {}
for e in ENGINES:
    name = e["name"]
    sub = tps_df[tps_df["engine"] == name]
    best = sub.loc[sub["cards/sec"].idxmax()]
    # A cloud vCPU is one hardware thread, so divide by LOGICAL cores.
    per_vcpu[name] = best["cards/sec"] / best["workers"]

cost_rows = []
for e in ENGINES:
    name = e["name"]
    rate = per_vcpu[name]
    rss = hw_df.loc[name, "peak RSS / worker MB"]
    for target in TPS_TARGETS:
        vcpus = target / (rate * UTILISATION)
        for srv in SERVERS:
            by_cpu = math.ceil(vcpus / srv["vcpu"])
            fit_ram = max(1, math.floor(srv["ram_gb"] * 1024 / rss)) if rss and not np.isnan(rss) else srv["vcpu"]
            workers_per_server = min(srv["vcpu"], fit_ram)
            by_ram = math.ceil(vcpus / workers_per_server)
            servers = max(by_cpu, by_ram)
            monthly = servers * srv["usd_per_hour"] * HOURS_PER_MONTH
            cards_per_hour = target * 3600
            cost_rows.append({
                "engine": name, "target TPS": target, "server": srv["name"],
                "cards/sec per vCPU": round(rate, 2),
                "workers/server": workers_per_server,
                "servers": servers,
                "limited by": "RAM" if by_ram > by_cpu else "CPU",
                "USD/month": round(monthly, 0),
                "USD/1000 cards": round(servers * srv["usd_per_hour"] / cards_per_hour * 1000, 4),
            })

cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(OUTDIR / "cost.csv", index=False, encoding="utf-8-sig")

for srv in SERVERS:
    print(f"\n{srv['name']} — {srv['vcpu']} vCPU, {srv['ram_gb']} GB, "
          f"${srv['usd_per_hour']}/hr")
    display(cost_df[cost_df["server"] == srv["name"]]
            .pivot(index="engine", columns="target TPS", values=["servers", "USD/month"])
            .reindex([e["name"] for e in ENGINES]))

## 7. Summary

One row per engine. Accuracy first, because it is the filter: rule out anything that
cannot read NID numbers reliably, then compare what is left on speed and cost.

In [ ]:
cheapest = (cost_df[cost_df["target TPS"] == TPS_TARGETS[0]]
            .sort_values("USD/month").groupby("engine").first())

summary = pd.DataFrame({
    "NID exact":     exact["nid"].map(_fmt_pct),
    "DOB exact":     exact["dob"].map(_fmt_pct),
    "Bangla name":   exact["name_bn"].map(_fmt_pct),
    "English name":  exact["name_en"].map(_fmt_pct),
    "p50 ms":        lat_df["p50 ms"],
    "p95 ms":        lat_df["p95 ms"],
    "cards/s/vCPU":  pd.Series(per_vcpu).round(2),
    "RSS MB":        hw_df["peak RSS / worker MB"],
    "cold start s":  hw_df["cold start s"],
    f"servers @{TPS_TARGETS[0]} TPS": cheapest["servers"],
    "USD/1000 cards": cheapest["USD/1000 cards"],
}).reindex([e["name"] for e in ENGINES])

summary.to_csv(OUTDIR / "SUMMARY.csv", encoding="utf-8-sig")
print(f"Run {RUN_ID} — all results in {OUTDIR}\n")
summary

### What to write up

Work through these in order; the first one that rules an engine out saves you arguing
about the rest.

1. **Does it read NID and date of birth reliably enough?** These two carry the
   compliance risk. An engine that fails here is out, whatever it costs.
2. **Can it read Bangla at all?** PaddleOCR's fast models cannot. That is not a bug
   in this benchmark, it is the finding — and it is what makes a hybrid worth
   considering: one engine for Bangla names, another for digits.
3. **Were the failures the engine's fault?** Section 2.1 separates "returned nothing"
   from "read it wrong". Empty fields on otherwise-good OCR usually mean the parser
   missed a label, which is fixable without changing engines.
4. **Is p95 acceptable inside the sign-up flow?** A user is waiting. p50 is not the
   number that annoys them.
5. **What does 1000 TPS actually mean here?** Sustained, that is around 86 million
   cards a day — roughly half of Bangladesh's population, every day. It is almost
   certainly a peak-burst ceiling rather than a steady rate. Worth confirming the
   real peak with your supervisor: designing for 1000 TPS instead of 50 changes the
   cost answer by a factor of twenty.

**Known limits of this benchmark**, which belong in the report:

- 20 cards is enough to compare engines and to find obvious failure modes, but not
  enough for a precise accuracy figure. Widen to the full ~200-card set before
  quoting an accuracy number to anyone.
- Measured on one Windows laptop. Cloud vCPUs differ; rerun section 4 on one real
  instance before trusting the cost table.
- CPU only. If a GPU is on the table, run EasyOCR and PaddleOCR again with
  `BENCH_GPU=1` as separate, clearly labelled rows — GPU instances cost several times
  more, so they need their own cost line.
- Full-card OCR. Per-field cropping would likely improve accuracy for every engine
  and is the obvious next experiment.